# IDI — LoRA adapter training (Qwen2.5-Coder-3B, QLoRA via Unsloth)

Trains one agent adapter on the datasets built by `training/build_dataset.py` and exports
it as a **GGUF LoRA** that `llama-server` loads next to the Q4_K_M base model.

**Runtime**: free T4 is enough (3B in 4-bit ≈ 6 GB VRAM; ~600 examples × 3 epochs ≈ 20–40 min).

**Run this notebook twice**: once with `AGENT = "sql_generator"`, once with
`AGENT = "query_understanding"`.

**Getting the repo files in**: either set `REPO_URL` below (git clone), or upload the repo
folder to `/content/idi` via the Files pane — only `data/synthetic/` and `training/` are needed.

Recipe (r=16, α=32, dropout=0.05, 3 epochs, lr 2e-4, completion-only loss) lives in
`training/lora_config.py` — edit it there, not here.

In [ ]:
AGENT = "sql_generator"  # or "query_understanding"
REPO_URL = ""  # e.g. "https://github.com/<user>/intelligent-database-interface.git"; leave empty if uploading manually

In [ ]:
%%capture
!pip install unsloth
!pip install sqlglot datasets

In [ ]:
import json, os, sys

if REPO_URL and not os.path.isdir("/content/idi"):
    !git clone --depth 1 {REPO_URL} /content/idi
REPO = "/content/idi"
assert os.path.isdir(f"{REPO}/data/synthetic"), (
    "Repo files not found. Set REPO_URL or upload the repo (data/synthetic + training) to /content/idi."
)
sys.path.insert(0, REPO)

from training import lora_config as cfg
from training.gretel_mix import build_gretel_mix

paths = cfg.ADAPTERS[AGENT]
with open(f"{REPO}/data/synthetic/{paths['train_file']}", encoding="utf-8") as f:
    train_records = [json.loads(l) for l in f]
with open(f"{REPO}/data/synthetic/{paths['eval_file']}", encoding="utf-8") as f:
    eval_records = [json.loads(l) for l in f]

# The exact system prompt the runtime agent uses — shared with the gretel mix.
SYSTEM_PROMPT = train_records[0]["messages"][0]["content"]
print(f"{AGENT}: {len(train_records)} soundwave train / {len(eval_records)} eval examples")

In [ ]:
# External mix: gretelai/synthetic_text_to_sql (medium/hard SELECTs only) so the
# adapter keeps general SQL competence. Soundwave remains the majority.
if paths["gretel_mix_examples"]:
    from datasets import load_dataset
    raw = load_dataset("gretelai/synthetic_text_to_sql", split="train")
    mix = build_gretel_mix(raw, paths["gretel_mix_examples"], SYSTEM_PROMPT,
                           seed=cfg.TRAINING["seed"])
    train_records = train_records + mix
    print(f"added {len(mix)} gretel examples -> {len(train_records)} total")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.BASE_MODEL,
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, **cfg.LORA)

In [ ]:
import random
from datasets import Dataset

random.Random(cfg.TRAINING["seed"]).shuffle(train_records)
ds = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                            add_generation_prompt=False)}
    for r in train_records
])
print(ds[0]["text"][:600])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

common = dict(output_dir=f"outputs_{AGENT}", dataset_text_field="text",
              report_to="none", **cfg.TRAINING)
try:  # trl renamed max_seq_length -> max_length across versions
    sft_args = SFTConfig(max_seq_length=cfg.MAX_SEQ_LENGTH, **common)
except TypeError:
    sft_args = SFTConfig(max_length=cfg.MAX_SEQ_LENGTH, **common)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=sft_args)

# Loss on the assistant completion only — never on the schema-heavy prompt.
trainer = train_on_responses_only(
    trainer,
    instruction_part=cfg.INSTRUCTION_PART,
    response_part=cfg.RESPONSE_PART,
)
trainer.train()

In [ ]:
# Held-out eval — a strict-ish proxy. Normalized exact match undercounts real quality
# (different-but-equivalent SQL scores 0); the real gate is `python tests/evaluate.py`
# against the live backend once the GGUF is installed locally.
import re

FastLanguageModel.for_inference(model)

def generate(messages, max_new_tokens=400):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def norm_sql(sql):
    return re.sub(r"\s+", " ", sql).strip().rstrip(";").lower()

N_EVAL = min(40, len(eval_records))
hits, results = 0, []
for rec in eval_records[:N_EVAL]:
    reply = generate(rec["messages"][:2])
    if AGENT == "sql_generator":
        m = re.search(r"```sql\s*([\s\S]*?)```", reply, re.IGNORECASE)
        pred = m.group(1) if m else reply
        ok = norm_sql(pred) == norm_sql(rec["meta"]["gold_sql"])
    else:
        try:
            data = json.loads(re.search(r"\{[\s\S]+\}", reply).group())
            gold = json.loads(rec["messages"][2]["content"])
            gold_entities = {e.lower() for e in gold["entities"]}
            pred_entities = {str(e).lower() for e in data.get("entities", [])}
            ok = bool(data.get("plain_restatement")) and (
                not gold_entities or bool(gold_entities & pred_entities)
            )
        except Exception:
            ok = False
    hits += int(ok)
    results.append((rec["meta"]["id"], bool(ok)))

print(f"\n{AGENT} eval: {hits}/{N_EVAL} ({hits/N_EVAL:.0%})")
for rid, ok in results:
    print(" OK " if ok else "FAIL", rid)

In [ ]:
# Export: save the LoRA, convert to GGUF with llama.cpp, download.
LORA_DIR = f"lora_{AGENT}"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

if not os.path.isdir("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q gguf sentencepiece

OUT_GGUF = f"/content/{AGENT}.gguf"
!python /content/llama.cpp/convert_lora_to_gguf.py {LORA_DIR} \
    --base-model-id {cfg.BASE_MODEL_FOR_GGUF} --outtype f16 --outfile {OUT_GGUF}

from google.colab import files
files.download(OUT_GGUF)

## Local integration (back on your machine)

1. Place the downloaded file at `adapters/<agent>.gguf`, i.e. `adapters/sql_generator.gguf`
   and `adapters/query_understanding.gguf` (gitignored, like `models/`).
2. `adapters/registry.json` already declares `kind: "gguf"` for both agents — no edit needed.
   Missing GGUF files fall back to the `.md` instruction profiles automatically.
3. Restart everything with `python start.py` — it passes one `--lora` flag per GGUF found
   in `adapters/`, and the orchestrator activates the right adapter scale per agent.
4. Verify against the pre-LoRA baseline (5/7 strict, `data/benchmarks/eval_2026-07-14.md`):
   ```
   python tests/evaluate.py     # target: >= 6/7 strict, EC-02 and EC-04 green
   python tests/ab_harness.py   # base vs adapter lift
   ```
5. Anti-memorization sanity: ask 3–5 held-out eval wordings through the UI (`/query`) —
   they were never trained on.